# Building a Custom Classifier

In [105]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [106]:
url = "https://raw.githubusercontent.com/clairett/pytorch-sentiment-classification/master/data/SST2/train.tsv"
data = pd.read_csv(url, sep="\t", names=["text", "sentiment"])
data["sentiment"] = data["sentiment"].map({0: "negative", 1: "positive"})
data["text"] = data.apply(lambda x: re.sub(r"^[\w\s]", "", x["text"]), axis=1)
print(data[0:5])

                                                text sentiment
0   stirring , funny and finally transporting re ...  positive
1  pparently reassembled from the cutting room fl...  negative
2  hey presume their audience wo n't sit still fo...  negative
3  his is a visually stunning rumination on love ...  positive
4  onathan parker 's bartleby should have been th...  positive


In [107]:
data = data.sample(frac=1).reset_index(drop=True)

In [108]:
X = data['text']
y = data['sentiment']

In [109]:
tfidfvec = TfidfVectorizer()
tfidfvec_fit = tfidfvec.fit_transform(X)
tfidf_bag = pd.DataFrame(tfidfvec_fit.toarray(), columns = tfidfvec.get_feature_names_out())

In [110]:
tfidf_bag

,00,000,10,100,101,103,105,10th,11,110,...,zishe,ziyi,zoe,zombie,zone,zoning,zpetek,zumaki,zwick,zzzzzzzzz
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6915,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6916,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6917,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6918,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [111]:
# split into train and test data
X_train, X_test, y_train, y_test = train_test_split(tfidf_bag, y, test_size=0.3, random_state = 7)

## Logistic Regression

In [112]:
# train the model
lr = LogisticRegression(random_state=1).fit(X_train, y_train)

In [113]:
# generate predictions
y_pred_lr = lr.predict(X_test)

In [114]:
# evaluate how the model performed
accuracy_score(y_pred_lr, y_test)

0.7586705202312138

In [115]:
y_pred_lr

array(['positive', 'positive', 'positive', ..., 'positive', 'negative',
       'positive'], dtype=object)

In [116]:
y_test

3194    positive
5258    positive
4537    positive
3324    positive
5071    positive
          ...   
1310    positive
1674    negative
4809    negative
5752    positive
227     negative
Name: sentiment, Length: 2076, dtype: object

In [ ]:
print(classification_report(y_test, y_pred_lr, zero_division=0))
# precision: of all the reviews that were predicted to be positive, how many were actually positive?
# recall: of all the reviews that were actually positive, how many were predicted to be positive
# The F1 score tells you how well your model balances being correct when it predicts something (precision) and not missing things it should find (recall). A high F1 score means your model is good at both: it makes few mistakes and doesn’t miss many correct answers.
# support: the number of occurrences of each class in the test set. It is used to calculate the precision, recall, and F1 score for each class. A high support indicates that there are many examples of that class in the test set, which can lead to more accurate predictions for that class.

              precision    recall  f1-score   support

    negative       0.77      0.71      0.74      1000
    positive       0.75      0.81      0.78      1076

    accuracy                           0.76      2076
   macro avg       0.76      0.76      0.76      2076
weighted avg       0.76      0.76      0.76      2076



## Naive Bayes

In [141]:
# classification algorithm that uses probabilities to make predictions
from sklearn.naive_bayes import MultinomialNB

In [ ]:
nb = MultinomialNB().fit(X_train, y_train)

In [132]:
y_pred_nb = nb.predict(X_test)

In [133]:
accuracy_score(y_pred_nb, y_test)

0.7721579961464354

In [134]:
print(classification_report(y_test, y_pred_nb, zero_division=0))

              precision    recall  f1-score   support

    negative       0.81      0.69      0.74      1000
    positive       0.75      0.85      0.79      1076

    accuracy                           0.77      2076
   macro avg       0.78      0.77      0.77      2076
weighted avg       0.78      0.77      0.77      2076



## Linear Support Vector Machine

In [135]:
from sklearn.linear_model import LogisticRegression, SGDClassifier

In [136]:
svm = SGDClassifier().fit(X_train, y_train)
# possible hyper params, loss function, regularization

In [137]:
y_pred_svm = svm.predict(X_test)

In [138]:
accuracy_score(y_pred_svm, y_test)

0.7697495183044316

In [139]:
print(classification_report(y_test, y_pred_svm, zero_division=0))

              precision    recall  f1-score   support

    negative       0.76      0.77      0.76      1000
    positive       0.78      0.77      0.78      1076

    accuracy                           0.77      2076
   macro avg       0.77      0.77      0.77      2076
weighted avg       0.77      0.77      0.77      2076



In [140]:
#TODO: try with tfidf vectorizer, and with ngrams
#TODO clean the text and see if it improves performance
#TODO: try with a larger dataset and see if it improves performance